# Chapitre 7 - Cours 5 : pourquoi sharder ? Avantages, inconvénients et multi-location

Les cours 1 à 4 ont répondu à la question « comment sharder ? ». Ce dernier cours prend
du recul et pose deux questions que la 2e édition du livre traite en ouverture du
chapitre : **faut-il sharder**, et à quel prix ? Et le sharding peut-il servir à autre
chose qu'à passer à l'échelle, en particulier à **isoler les clients** d'un logiciel
multi-locataire (SaaS) ? Il couvre les sections *Pros and Cons of Sharding* et
*Sharding for Multitenancy*.

**Plan.**

1. Quand une seule machine suffit : ordres de grandeur.
2. Ce que le sharding apporte : des écritures qui passent à l'échelle.
3. Le prix : clé de partitionnement et transactions multi-shards.
4. Sharder à l'intérieur d'une machine : un processus par cœur.
5. Multi-location : un locataire, un shard ?
6. Isolation : voisin bruyant, permissions et architecture cellulaire.
7. Sauvegarde, effacement, résidence et migrations progressives.
8. Les difficultés : locataires qui grandissent, géants et requêtes transverses.

In [1]:
import sqlite3
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import partitioning as P
from ddia import partitioning_tenants as T
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET, COLORS,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
rng = np.random.default_rng(0)
fmt = lambda v: f"{v:,.0f}".replace(",", " ")      # 1234567 -> « 1 234 567 »